# Extract The Attributes of The Generated Images

In [ ]:
import pandas as pd
import textwrap

import tomllib
import avoddiag as ag

## Load Configuration

In [ ]:
with open("config/config.toml", "rb") as f:
    config = tomllib.load(f)

## Load Image Dataset

In [ ]:
image_dataset = ag.image.data.Dataset(
    root_folder_path="output/TestDataset_01_Generated_Images"
)

In [ ]:
import random

# image_file_name = random.choice(image_dataset.image_file_names)
image_data = random.choice(image_dataset) #[image_file_name]
image = image_data["image"]
image


## Extract Image Attributes

### Gemini

In [ ]:
# Initialize the Google GenAI provider
provider_google = ag.providers.google.GoogleGenAI(api_key=config['providers']['google']['api_key'])

In [ ]:
provider_google.list_models()

In [ ]:
model_name = "gemini-2.5-flash"
# model_name = "gemini-3.5-flash"
model_config = ag.providers.google.types.GenerateContentConfig(
    temperature=0.0,
    thinking_config=ag.providers.google.types.ThinkingConfig(
        thinking_budget=0,
    )
)

query_metadata = []
for image_data in image_dataset:
    prompt = textwrap.dedent(
        f"""Extract the following attributes from the image and return them as a JSON object. If an attribute cannot be determined from the image, set its value to "N/A". Answer with a single expression per attribute.

        {{
          "camera_view_confirmation": "The answer of the following question: Is the camera view depicted in this image a top-down vertical (a.k.a. nadir) view? Answer with yes or no.",
          "scene_type": "string describing the environment",
          "scene_concepts": ["a list of concepts describing the scene in the image"],
          "scene_type_confirmation": "The answer of the following question: Can the environment depicted in this image be described as a {image_data['metadata']['attributes_input']['attribute:scene_type']} environment? Answer with yes or no.",
          "season": "string indicating the season, selected from the following list ['spring', 'summer', 'fall', 'winter'] or N/A",
          "season_confirmation": "The answer of the following question: Can the season depicted in this image be described as {image_data['metadata']['attributes_input']['attribute:season']}? Answer with yes or no.",
          "weather": "string describing the weather conditions or N/A",
          "weather_confirmation": "The answer of the following question: Can the weather conditions depicted in this image be described as {image_data['metadata']['attributes_input']['attribute:weather']}? Answer with yes or no.",
          "vehicle_presence": "The answer of the following question: Are there any vehicles in this image? Answer with yes or no.",
          "vehicle_count": "number indicating the count of vehicles in the image or N/A",
          "vehicle_colors": "a list of all unique colors of vehicles in the image or N/A"
        }}

        Example:
        Input: 
        Output: {{
          "camera_view_confirmation": "yes",
          "scene_type": "urban",
          "scene_concepts": ["urban environment", "trees", "parking lot"],
          "scene_type_confirmation": "yes",
          "season": "autumn",
          "season_confirmation": "yes",
          "weather": "overcast",
          "weather_confirmation": "yes",
          "vehicle_presence": "yes",
          "vehicle_count": 5,
          "vehicle_colors": ["red", "blue", "green"]
        }}"""
    )
    
    query_metadata.append(
        {
            'image_file_name': image_data['image_file_name'],
            'prompt': prompt,
        }
    )

query_metadata = pd.DataFrame(query_metadata)
query_metadata


In [ ]:
attribute_extractor = ag.image.analysis.AttributeExtractor(
    provider=provider_google,
    model_name=model_name,
    with_caching=True,
    config=model_config,
)

In [ ]:
attribute_extractor.extract_dataset_attributes(
    image_dataset=image_dataset,
    query_metadata=query_metadata,
)

In [ ]:
# Display the extracted attributes data
metadata_key = f'attributes_generated:{model_name}'
print(f'Visualizing metadata key: {metadata_key}')
image_dataset.metadata[metadata_key]
# image_dataset.metadata[metadata_key]['attribute:scene_type_confirmation'].value_counts()
# (image_dataset.metadata[metadata_key]['attribute:scene_type_confirmation']=='no').sum()

### Moondream2

In [ ]:
!nvidia-smi

In [ ]:
# Initialize the Moondream provider
provider_moondream = ag.providers.transformers.Moondream2(
    gpu_ids=[0,1,2,3,4,5,6,7],  # Specify GPU IDs if available
)

In [ ]:
model_name = "vikhyatk/moondream2"
model_revision = "2025-06-21"

attribute_extractor = ag.image.analysis.AttributeExtractor(
    provider=provider_moondream,
    model_name=model_name,
    model_revision=model_revision
)

In [ ]:
query_metadata = []
for image_data in image_dataset:
    prompt = textwrap.dedent(
        f"""Extract the following attributes from the image and return them as a JSON object. If an attribute cannot be determined from the image, set its value to "N/A". Answer with a single expression or a list of expression for each attribute. 
        Return a dictiornary with the following keys:

        {{
          "camera_view_confirmation": "The answer of the following question: Is the camera view depicted in this image a top-down vertical (a.k.a. nadir) view? Answer with yes or no.",
          "scene_type": "string describing the environment",
          "scene_concepts": "a list of concepts describing the scene in the image",
          "scene_type_confirmation": "The answer of the following question: Can the environment depicted in this image be described as a {image_data['metadata']['attributes_input']['attribute:scene_type']} environment? Answer with yes or no.",
          "season": "string indicating the season, selected from the following list ['spring', 'summer', 'fall', 'winter'] or N/A",
          "season_confirmation": "The answer of the following question: Can the season depicted in this image be described as {image_data['metadata']['attributes_input']['attribute:season']}? Answer with yes or no.",
          "weather": "string describing the weather conditions or N/A",
          "weather_confirmation": "The answer of the following question: Can the weather conditions depicted in this image be described as {image_data['metadata']['attributes_input']['attribute:weather']}? Answer with yes or no.",
          "vehicle_presence": "The answer of the following question: Are there any vehicles in this image? Answer with yes or no.",
          "vehicle_count": "number indicating the count of vehicles in the image or N/A",
          "vehicle_colors": "a list of all unique colors of vehicles in the image or N/A"
        }}"""
    )

    query_metadata.append(
        {
            'image_file_name': image_data['image_file_name'],
            'prompt': prompt,
        }
    )

query_metadata = pd.DataFrame(query_metadata)
query_metadata


In [ ]:
attribute_extractor.extract_dataset_attributes(
    image_dataset=image_dataset,
    query_metadata=query_metadata,
)

In [ ]:
metadata_key = f'attributes_generated:{model_name.replace("/", "+")}'
print(metadata_key)
image_dataset.metadata[metadata_key]